# Confidence and OOD gating

This notebook tests whether a reflex can defer unfamiliar inputs. The PCA gate is intentionally simple. It is inspired by the trusted-space question in [z-manifold](https://github.com/infinition/z-manifold), but it does not reproduce the adapter-space method from [arXiv:2607.05300](https://arxiv.org/abs/2607.05300).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from paradigm import ReflexCompiler, TrustedSubspaceGate
from paradigm.synthetic import make_traces

train = make_traces(4000, seed=2)
X = np.stack([t.features for t in train])
reflex = ReflexCompiler(random_state=2).fit(train)
gate = TrustedSubspaceGate(variance=0.95, quantile=0.99).fit(X)

rng = np.random.default_rng(2)
X_iid = rng.uniform(0, 1, size=(1000, X.shape[1]))
X_ood = rng.uniform(1.25, 2.0, size=(1000, X.shape[1]))
print("IID accept:", gate.accept(X_iid).mean())
print("OOD accept:", gate.accept(X_ood).mean())

## Controls to add

Compare the PCA residual with k-nearest-neighbor distance, Mahalanobis distance, ordinary confidence, and a nonlinear autoencoder. Include a weak-pool split containing valid behavior absent from the trusted set.